# GBE reward intervals

Build one row per pair of successive happiness ratings, keeping the reward events between them in order.

Use development participants and versions 3–5. Keep all plays and ratings for now. This table will support the matching-versus-reward-history check and later feasibility modelling.

The selection check below asks whether matching varies with prior mood and recent reward exposure.

## 1. Load the data and apply development split

Same file and participant split as notebook 3. Load the full file, then select development IDs before constructing any intervals.

In [1]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat

PROJECT_DIR = Path.cwd().parents[1]
DATA_PATH = PROJECT_DIR / "data" / "raw" / "gbe" / "Rutledge_GBE_risk_data_TOD.mat"

print("Data file found:", DATA_PATH.is_file())

Data file found: True


In [2]:
mat = loadmat(DATA_PATH, simplify_cells=True)
participants = mat["subjData"]

print("People:", len(participants))

People: 47067


In [3]:
def get_plays(participant):
    # same container handling as notebook 2
    if participant["nPlays"] == 1:
        return [participant["data"]]
    return list(participant["data"])

In [4]:
import hashlib

SPLIT_PATH = Path.cwd().parent / "splits" / "gbe_split_2026-10-07.csv"
EXPECTED_SPLIT_SHA256 = "861de3c444f506c76cbac3208840fd62d57a71b6aad63fceb83a802d8da139e9"

assert hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = pd.read_csv(SPLIT_PATH)

assert list(split.columns) == ["id", "set"]
assert pd.api.types.is_integer_dtype(split["id"])
assert split["id"].is_unique
assert set(split["set"]) == {"development", "holdout"}

# make sure no IDs are missing or invented relative to the loaded release
loaded_ids = [int(person["id"]) for person in participants]
assert len(loaded_ids) == len(set(loaded_ids))
assert set(split["id"]) == set(loaded_ids)

development_ids = set(split.loc[split["set"] == "development", "id"])
holdout_ids = set(split.loc[split["set"] == "holdout", "id"])
assert development_ids.isdisjoint(holdout_ids)
assert len(development_ids) == 10_000
assert len(holdout_ids) == 37_067

# leave participants unchanged; keep the released ordering in this subset
development_participants = [
    person for person in participants if int(person["id"]) in development_ids
]
assert len(development_participants) == 10_000

print("Split checksum and IDs verified.")
print("Development people:", len(development_participants))
print("Reserved hold-out IDs:", len(holdout_ids))

Split checksum and IDs verified.
Development people: 10000
Reserved hold-out IDs: 37067


## 2. Pick and inspect one play

Start with one interval to check the boundaries. Baseline is stored in the first trial row, but comes before any outcomes. The next rating follows trial 3, so this interval should contain trials 1–3.

In [5]:
for participant in development_participants:
    versions = np.atleast_1d(participant["designVersion"])

    eligible = np.flatnonzero(np.isin(versions, [3, 4, 5]))
    if len(eligible):
        play_index = int(eligible[0])
        example_participant = participant
        example_play = get_plays(participant)[play_index]
        example_version = int(versions[play_index])
        break

print("Participant:", example_participant["id"])
print("Available play:", play_index + 1)
print("Version:", example_version)

Participant: 128
Available play: 2
Version: 4


In [6]:
rating_rows = np.flatnonzero(~np.isnan(example_play[:, 9]))
boundaries = example_play[rating_rows, 0].astype(int).copy() # get the trial numbers
boundaries[0] = 0  # baseline precedes trial 1

left, right = boundaries[:2] # 0 and whatever the first rating trial number is
events = example_play[left:right]

print("Starting happiness:", example_play[rating_rows[0], 9])
print("Ending happiness:", example_play[rating_rows[1], 9])

pd.DataFrame(
    events[:, [0, 2, 3, 4, 6, 7]],
    columns=[
        "trial", "certainValue", "winValue",
        "loseValue", "choseRisky", "outcome",
    ],
)

Starting happiness: 52.0
Ending happiness: 50.0


,trial,certainValue,winValue,loseValue,choseRisky,outcome
0,1.0,55.0,124.0,0.0,1.0,124.0
1,2.0,0.0,55.0,-37.0,1.0,55.0
2,3.0,0.0,40.0,-28.0,1.0,-28.0


### Reconstruct the reward terms

All three example choices are gambles. EVs are 62, 9 and 6; RPEs are 62, 46 and −34. Check those by hand before generalising.

In [7]:
offered_ev = (events[:, 3] + events[:, 4]) / 2
chose_gamble = events[:, 6] == 1

certain_reward = np.where(chose_gamble, 0, events[:, 2])
gamble_ev = np.where(chose_gamble, offered_ev, 0)
rpe = np.where(chose_gamble, events[:, 7] - offered_ev, 0)

np.testing.assert_allclose(gamble_ev, [62, 9, 6])
np.testing.assert_allclose(rpe, [62, 46, -34])
np.testing.assert_allclose(certain_reward, [0, 0, 0])

# keep both signed and absolute sums as candidates for now
print("Sum of RPEs:", rpe.sum())
print("Sum of absolute RPEs:", np.abs(rpe).sum())

Sum of RPEs: 74.0
Sum of absolute RPEs: 142.0


### One interval row

Keep both rating endpoints, their slider starts, the interval boundaries and the reward terms. A row represents the events after the prior rating through the outcome before the next rating.

In [8]:
start_row, end_row = rating_rows[:2]

example_interval = {
    "id": int(example_participant["id"]),
    "available_play": play_index + 1,
    "version": example_version,
    "interval": 1,
    "start_boundary": int(left),
    "end_boundary": int(right),
    "n_trials": len(events),
    "prior_happiness": example_play[start_row, 9],
    "next_happiness": example_play[end_row, 9],
    "prior_startValue": example_play[start_row, 10],
    "next_startValue": example_play[end_row, 10],
    "prior_matches_start": bool(np.isclose(
        example_play[start_row, 9],
        example_play[start_row, 10],
        rtol=0, atol=1e-10,
    )),
    "next_matches_start": bool(np.isclose(
        example_play[end_row, 9],
        example_play[end_row, 10],
        rtol=0, atol=1e-10,
    )),
    "sum_rpe": rpe.sum(),
    "sum_abs_rpe": np.abs(rpe).sum(),
    "sum_certain_reward": certain_reward.sum(),
    "sum_gamble_ev": gamble_ev.sum(),
    "trial_numbers": events[:, 0].astype(int).tolist(),
    "rpe_sequence": rpe.tolist(),
    "certain_reward_sequence": certain_reward.tolist(),
    "gamble_ev_sequence": gamble_ev.tolist(),
}

pd.DataFrame([example_interval]).T

,0
id,128
available_play,2
version,4
interval,1
start_boundary,0
end_boundary,3
n_trials,3
prior_happiness,52.0
next_happiness,50.0
prior_startValue,27.0


## 3. Build the interval table

### Try all 11 intervals in the example play

Put the same calculations in a function, then check that the intervals join up. Keep the sequences too, so the sums aren’t the only version of reward history left.

In [9]:
# return a list of interval dictionaries for specified play, participant, and version
def build_play_intervals(play, participant_id, available_play, version):
    rating_rows = np.flatnonzero(~np.isnan(play[:, 9]))

    boundaries = play[rating_rows, 0].astype(int).copy()
    boundaries[0] = 0  # baseline comes before any trials

    intervals = []

    for index in range(len(rating_rows) - 1):
        start_row = rating_rows[index]
        end_row = rating_rows[index + 1]
        left = boundaries[index]
        right = boundaries[index + 1]
        
        events = play[left:right]

        offered_ev = (events[:, 3] + events[:, 4]) / 2
        chose_gamble = events[:, 6] == 1

        certain_reward = np.where(chose_gamble, 0, events[:, 2])
        gamble_ev = np.where(chose_gamble, offered_ev, 0)
        rpe = np.where(chose_gamble, events[:, 7] - offered_ev, 0)

        interval = {
            "id": int(participant_id),
            "available_play": available_play,
            "version": version,
            "interval": index + 1,
            "start_boundary": int(left),
            "end_boundary": int(right),
            "n_trials": len(events),
            "prior_happiness": play[start_row, 9],
            "next_happiness": play[end_row, 9],
            "prior_startValue": play[start_row, 10],
            "next_startValue": play[end_row, 10],
            "prior_matches_start": bool(np.isclose(
                play[start_row, 9],
                play[start_row, 10],
                rtol=0, atol=1e-10,
            )),
            "next_matches_start": bool(np.isclose(
                play[end_row, 9],
                play[end_row, 10],
                rtol=0, atol=1e-10,
            )),
            "sum_rpe": rpe.sum(),
            "sum_abs_rpe": np.abs(rpe).sum(),
            "sum_certain_reward": certain_reward.sum(),
            "sum_gamble_ev": gamble_ev.sum(),
            "trial_numbers": events[:, 0].astype(int).tolist(),
            "rpe_sequence": rpe.tolist(),
            "certain_reward_sequence": certain_reward.tolist(),
            "gamble_ev_sequence": gamble_ev.tolist(),
        }
        intervals.append(interval)

    return intervals

example_intervals = build_play_intervals(
    example_play,
    example_participant["id"],
    play_index + 1,
    example_version,
)

print("Intervals:", len(example_intervals))
pd.DataFrame(example_intervals).head()

Intervals: 11


,id,available_play,version,interval,start_boundary,end_boundary,n_trials,prior_happiness,next_happiness,prior_startValue,...,prior_matches_start,next_matches_start,sum_rpe,sum_abs_rpe,sum_certain_reward,sum_gamble_ev,trial_numbers,rpe_sequence,certain_reward_sequence,gamble_ev_sequence
0,128,2,4,1,0,3,3,52.0,50.0,27.0,...,False,False,74.0,142.0,0.0,77.0,"[1, 2, 3]","[62.0, 46.0, -34.0]","[0.0, 0.0, 0.0]","[62.0, 9.0, 6.0]"
1,128,2,4,2,3,6,3,50.0,52.0,33.0,...,False,False,76.5,76.5,35.0,22.5,"[4, 5, 6]","[0.0, 47.0, 29.5]","[35.0, 0.0, 0.0]","[0.0, -7.0, 29.5]"
2,128,2,4,3,6,9,3,52.0,29.0,32.0,...,False,False,-164.5,164.5,0.0,145.5,"[7, 8, 9]","[-37.0, -55.5, -72.0]","[0.0, 0.0, 0.0]","[18.0, 55.5, 72.0]"
3,128,2,4,4,9,11,2,29.0,31.0,37.0,...,False,False,24.0,74.0,0.0,64.0,"[10, 11]","[-25.0, 49.0]","[0.0, 0.0]","[15.0, 49.0]"
4,128,2,4,5,11,14,3,31.0,39.0,48.0,...,False,False,6.0,156.0,0.0,-22.0,"[12, 13, 14]","[29.5, -75.0, 51.5]","[0.0, 0.0, 0.0]","[29.5, -75.0, 23.5]"


In [10]:
# flatten the trial lists, keeping their order
covered_trials = [
    trial
    for interval in example_intervals
    for trial in interval["trial_numbers"]
]

# sanity checks: 11 intervals, trials 1–30 covered once, in order
assert len(example_intervals) == 11
np.testing.assert_array_equal(covered_trials, np.arange(1, 31))

# rating endpoints join correctly
for previous, following in zip(example_intervals[:-1], example_intervals[1:]):
    assert previous["end_boundary"] == following["start_boundary"]
    assert previous["next_happiness"] == following["prior_happiness"]
    assert previous["next_startValue"] == following["prior_startValue"]

for interval in example_intervals:
    for field in [
        "trial_numbers", "rpe_sequence",
        "certain_reward_sequence", "gamble_ev_sequence",
    ]:
        assert len(interval[field]) == interval["n_trials"] # sequence lengths match

print("11 intervals; trials 1–30 covered once, in order.")
print("Rating endpoints join correctly; sequence lengths match.")

11 intervals; trials 1–30 covered once, in order.
Rating endpoints join correctly; sequence lengths match.


### Apply to development plays

Repeat the boundary checks for each play. No trials skipped or counted twice, and no plays or ratings removed. Expect 6,675 people, 11,783 plays and 129,613 intervals.

In [11]:
interval_rows = []
plays_checked = 0

for participant in development_participants:
    versions = np.atleast_1d(participant["designVersion"])

    for play_index, play in enumerate(get_plays(participant)):
        version = int(versions[play_index])
        if version not in (3, 4, 5):
            continue

        intervals = build_play_intervals(
            play, participant["id"], play_index + 1, version
        )

        # identify the play if a check fails
        play_key = (int(participant["id"]), play_index + 1)
        assert len(intervals) == 11, play_key

        covered_trials = [
            trial
            for interval in intervals
            for trial in interval["trial_numbers"]
        ]
        np.testing.assert_array_equal(
            covered_trials, np.arange(1, 31),
            err_msg=f"Trial coverage: {play_key}",
        )

        for previous, following in zip(intervals[:-1], intervals[1:]):
            assert previous["end_boundary"] == following["start_boundary"], play_key
            assert previous["next_happiness"] == following["prior_happiness"], play_key
            assert previous["next_startValue"] == following["prior_startValue"], play_key

        for interval in intervals:
            for field in [
                "trial_numbers", "rpe_sequence",
                "certain_reward_sequence", "gamble_ev_sequence",
            ]:
                assert len(interval[field]) == interval["n_trials"], play_key

        interval_rows.extend(intervals)
        plays_checked += 1

development_intervals = pd.DataFrame(interval_rows)

assert not development_intervals.duplicated(
    ["id", "available_play", "interval"]
).any()

print("People:", development_intervals["id"].nunique())
print("Plays checked:", plays_checked)
print("Intervals:", len(development_intervals))

People: 6675
Plays checked: 11783
Intervals: 129613


### Check the stored summaries

Check that each interval’s sums match its sequences. This catches inconsistencies in the table, but isn’t an independent check of the RPE formula.

In [12]:
for row in development_intervals.itertuples(index=False):
    key = (row.id, row.available_play, row.interval)

    np.testing.assert_allclose(
        row.sum_rpe, np.sum(row.rpe_sequence),
        err_msg=f"RPE sum: {key}",
    )
    np.testing.assert_allclose(
        row.sum_abs_rpe, np.abs(row.rpe_sequence).sum(),
        err_msg=f"Absolute RPE sum: {key}",
    )
    np.testing.assert_allclose(
        row.sum_certain_reward, np.sum(row.certain_reward_sequence),
        err_msg=f"Certain reward sum: {key}",
    )
    np.testing.assert_allclose(
        row.sum_gamble_ev, np.sum(row.gamble_ev_sequence),
        err_msg=f"Gamble EV sum: {key}",
    )

print("All interval sums match their stored sequences.")

All interval sums match their stored sequences.


## 4. Matching and the preceding reward interval

Model whether the next rating matches its slider start, separately for
versions 3–5, using development participants only.

Main predictors:
- Signed and absolute RPE sums.
- Certain-reward and gamble-EV sums.
- Prior happiness and the next slider starting value, allowing curved relationships.
- Rating position as categories.

Keep all intervals and cluster uncertainty by participant. This is a
pooled comparison, so associations can reflect differences both within
and between people.

For one follow-up check, replace the two RPE sums with the final trial's
signed and absolute RPE. Keep the other predictors unchanged.

Check whether matching varies with prior happiness and interval reward exposure, allowing for slider start and rating position. This wil help assess whether removing matches would select a different set of reward–mood observations.

Signed RPE lets positive and negative surprises cancel; absolute RPE keeps their total size. These are simple interval summaries, not the final happiness model. Interval length follows a fixed rating-position schedule, so it isn’t added alongside the position indicators.

### Prepare the model data

Make a separate copy, code matching as 0/1 and scale rewards per 100 points. This changes coefficient units only. Also prepare the final-trial terms for the planned comparison.

In [13]:
# separate modelling copy
selection_data = development_intervals.copy()

selection_data["matches_next"] = (
    selection_data["next_matches_start"].astype(int)
)

reward_fields = [
    "sum_rpe", "sum_abs_rpe",
    "sum_certain_reward", "sum_gamble_ev",
]

# add per-100 versions of the reward fields for easier interpretation
for field in reward_fields:
    selection_data[f"{field}_per100"] = selection_data[field] / 100

selection_data["last_rpe_per100"] = (
    # take the final RPE from each interval sequence and divide by 100
    selection_data["rpe_sequence"].map(lambda values: values[-1]) / 100
)
selection_data["last_abs_rpe_per100"] = (
    selection_data["last_rpe_per100"].abs()
)

print("Intervals:", len(selection_data))
print("People:", selection_data["id"].nunique())

Intervals: 129613
People: 6675


### Build the predictor matrix

Use natural cubic splines with four df for prior happiness and slider start. Position is categorical. Stop on missing values rather than dropping rows silently, then check the matrix rank.

In [14]:
from patsy import dmatrices

selection_v3 = selection_data[
    selection_data["version"] == 3
].copy()

# patsy model description
# natural cubic spline with 4 df to account for potentially curved relationships
# treat interval as a categorical variable
selection_formula = """
matches_next ~
    sum_rpe_per100
    + sum_abs_rpe_per100
    + sum_certain_reward_per100
    + sum_gamble_ev_per100
    + cr(prior_happiness, df=4, constraints='center')
    + cr(next_startValue, df=4, constraints='center')
    + C(interval)
"""

y_selection, X_selection = dmatrices(
    selection_formula,
    data=selection_v3,
    return_type="dataframe",
    NA_action="raise", # if missing value stop, don't drop
)

assert len(X_selection) == len(selection_v3)
assert np.isfinite(X_selection.to_numpy()).all()

# count linearly independent predictor cols
rank = np.linalg.matrix_rank(X_selection.to_numpy())

print("Intervals:", len(X_selection))
print("Predictor columns:", X_selection.shape[1])
print("Matrix rank:", rank)

Intervals: 12804
Predictor columns: 23
Matrix rank: 23


### Fit version 3 first

Fit a generealised linear model for the binary match outcome. Cluster uncertainty by participant because each person can contribute lots of intervals. Unlike notebook 3’s start-position model, there are no play fixed effects here; this keeps all-match and no-match plays in the comparison.

In [15]:
import statsmodels.api as sm

selection_fit_v3 = sm.GLM(
    y_selection.iloc[:, 0], # outcome column
    X_selection, # the predictors
    family=sm.families.Binomial(), # binary outcome
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": selection_v3["id"], # cluster by participant ID
        "use_correction": True, # use standard small-sample correction
    },
)

print("Converged:", selection_fit_v3.converged)

Converged: True


### Read the reward terms

Odds ratios are per 100-point increase, holding the other predictors fixed. Below 1 means lower matching odds; above 1 means higher odds. These aren’t percentage-point changes in probability. Signed RPE is interpreted at a fixed absolute RPE sum.

Only the reward terms are displayed here. Prior happiness, starting value and position are still included in the fit.

In [16]:
reward_terms = [f"{field}_per100" for field in reward_fields]
confidence_intervals = selection_fit_v3.conf_int()

reward_results_v3 = pd.DataFrame({
    "term": reward_terms,
    "odds_ratio": np.exp(selection_fit_v3.params.loc[reward_terms]), #find the coeffs, convert from log-odds to odds ratio and apply odds-ratio scale
    "CI_low": np.exp(confidence_intervals.loc[reward_terms, 0]),
    "CI_high": np.exp(confidence_intervals.loc[reward_terms, 1]),
})

reward_results_v3.style.format({
    "odds_ratio": "{:.3f}",
    "CI_low": "{:.3f}",
    "CI_high": "{:.3f}",
}).hide(axis="index")

term,odds_ratio,CI_low,CI_high
sum_rpe_per100,0.980,0.919,1.045
sum_abs_rpe_per100,0.784,0.687,0.895
sum_certain_reward_per100,1.172,1.055,1.302
sum_gamble_ev_per100,0.985,0.907,1.071


Matching is less common after greater absolute RPE exposure, allowing for the other predictors. More positive certain-reward sums go with higher matching odds. Signed RPE and gamble EV show no clear direction here. So matching doesn’t look unrelated to the preceding reward interval in this version.

### Repeat across versions 3–5

Same formula and fitting method. Spline knots come from each version’s own data. Refit version 3 as a check that it matches the worked example above.

In [17]:
selection_fits = {}
reward_result_rows = []

for version in (3, 4, 5):
    data = selection_data[
        selection_data["version"] == version
    ].copy()

    y, X = dmatrices(
        selection_formula,
        data=data,
        return_type="dataframe",
        NA_action="raise",
    )

    assert len(X) == len(data)
    assert np.linalg.matrix_rank(X.to_numpy()) == X.shape[1]

    fit = sm.GLM(
        y.iloc[:, 0], X,
        family=sm.families.Binomial(),
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": data["id"],
            "use_correction": True,
        },
    )

    assert fit.converged, f"Version {version} did not converge"
    selection_fits[version] = fit
    ci = fit.conf_int()

    for term in reward_terms:
        reward_result_rows.append({
            "version": version,
            "term": term,
            "odds_ratio": np.exp(fit.params[term]),
            "CI_low": np.exp(ci.loc[term, 0]),
            "CI_high": np.exp(ci.loc[term, 1]),
        })

reward_results = pd.DataFrame(reward_result_rows)

np.testing.assert_allclose(
    selection_fits[3].params,
    selection_fit_v3.params,
    rtol=0, atol=1e-8,
)

reward_results.style.format({
    "odds_ratio": "{:.3f}",
    "CI_low": "{:.3f}",
    "CI_high": "{:.3f}",
}).hide(axis="index")

version,term,odds_ratio,CI_low,CI_high
3,sum_rpe_per100,0.980,0.919,1.045
3,sum_abs_rpe_per100,0.784,0.687,0.895
3,sum_certain_reward_per100,1.172,1.055,1.302
3,sum_gamble_ev_per100,0.985,0.907,1.071
4,sum_rpe_per100,0.978,0.929,1.030
4,sum_abs_rpe_per100,0.832,0.738,0.938
4,sum_certain_reward_per100,0.986,0.913,1.064
4,sum_gamble_ev_per100,0.976,0.914,1.042
5,sum_rpe_per100,0.957,0.933,0.981
5,sum_abs_rpe_per100,0.794,0.745,0.846


Across versions 3–5, matching odds are lower after larger absolute RPE sums. The other reward terms are smaller or less consistent. So dropping matches would change the mix of reward histories kept. This doesn’t tell me how much the eventual mood × RPE estimate would change.

## 5. Final-trial comparison

Replace the signed and absolute RPE sums with the final trial’s signed and absolute RPE. Keep certain-reward and gamble-EV sums, the curved terms and rating position unchanged.

This asks whether the pattern also appears for the most recent event. It doesn’t compare the last event’s contribution with earlier events directly.

In [18]:
recency_formula = selection_formula.replace(
    "sum_rpe_per100", "last_rpe_per100"
).replace(
    "sum_abs_rpe_per100", "last_abs_rpe_per100"
)

recency_terms = [
    "last_rpe_per100",
    "last_abs_rpe_per100",
    "sum_certain_reward_per100",
    "sum_gamble_ev_per100",
]

The recency fit should converge and have a full-rank matrix. Its coefficients should not be asserted equal to the interval-sum model: the RPE predictors are different.

In [19]:
recency_fits = {}
recency_result_rows = []

for version in (3, 4, 5):
    data = selection_data[
        selection_data["version"] == version
    ].copy()

    y, X = dmatrices(
        recency_formula,
        data=data,
        return_type="dataframe",
        NA_action="raise",
    )

    assert len(X) == len(data)
    assert np.linalg.matrix_rank(X.to_numpy()) == X.shape[1]

    fit = sm.GLM(
        y.iloc[:, 0], X,
        family=sm.families.Binomial(),
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": data["id"],
            "use_correction": True,
        },
    )

    assert fit.converged, f"Version {version} did not converge"
    recency_fits[version] = fit
    ci = fit.conf_int()
    
    for term in recency_terms:
        recency_result_rows.append({
            "version": version,
            "term": term,
            "odds_ratio": np.exp(fit.params[term]),
            "CI_low": np.exp(ci.loc[term, 0]),
            "CI_high": np.exp(ci.loc[term, 1]),
        })

recency_results = pd.DataFrame(recency_result_rows)

recency_results.style.format({
    "odds_ratio": "{:.3f}",
    "CI_low": "{:.3f}",
    "CI_high": "{:.3f}",
}).hide(axis="index")

version,term,odds_ratio,CI_low,CI_high
3,last_rpe_per100,0.950,0.861,1.050
3,last_abs_rpe_per100,0.734,0.616,0.875
3,sum_certain_reward_per100,1.153,1.038,1.279
3,sum_gamble_ev_per100,0.970,0.895,1.050
4,last_rpe_per100,0.932,0.862,1.007
4,last_abs_rpe_per100,0.698,0.591,0.825
4,sum_certain_reward_per100,0.975,0.901,1.054
4,sum_gamble_ev_per100,0.970,0.914,1.030
5,last_rpe_per100,0.926,0.888,0.966
5,last_abs_rpe_per100,0.723,0.663,0.788


The pattern also appears when using only the final trial’s RPE: larger absolute RPE goes with lower matching odds in all three versions. So the association isn’t limited to interval totals. This doesn’t yet show whether the last event matters more than earlier ones.

## 6. Conclusions and next steps

The development interval table covers all 30 trials once per play and keeps both ordered rewards and their summaries. No plays or ratings removed.

Matching varies with recent reward exposure in both specifications, most clearly with absolute RPE. That makes simply dropping matches a choice that needs testing, rather than an obvious clean-up step. These models mix differences within and between people and don’t establish why matching happens.

Next notebook: implement the happiness model described in Rutledge’s supplementary methods. Then use a provisional mood × RPE model in simulations to see what the slider mechanisms could do to recovery of a known effect, including a null.